# Excel aggregator: January + February + March 2026

Unified dataset from three monthly Excel reports for dashboard source v1.

What this notebook does:
- loads Excel files for Jan/Feb/Mar;
- aligns columns across months;
- adds `report_month` and basic normalized keys;
- aggregates into one final dataset;
- saves outputs to `.xlsx`, `.csv`, `.parquet`.

In [ ]:
import re
import warnings
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

warnings.filterwarnings(
    'ignore',
    category=FutureWarning,
    message='The behavior of DataFrame concatenation with empty or all-NA entries is deprecated.*'
)

pd.options.display.max_columns = None
pd.options.display.width = None
pd.options.display.max_colwidth = None

In [ ]:
# Parameters
sources = [
    {'report_month': '2026-01-01', 'excel_path': '/home/jovyan/documents/Equaring/Data/01_Январь_2026.xlsx', 'header': 1},
    {'report_month': '2026-02-01', 'excel_path': '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx', 'header': 0},
    {'report_month': '2026-03-01', 'excel_path': '/home/jovyan/documents/Equaring/Data/03_Март_2026.xlsx', 'header': 0},
]

excel_col_inn = 'ИНН'
excel_col_contract = 'Номер договора'

sa_type = 'SA'
mem_limit = '10g'

output_dir = '/home/jovyan/documents/Equaring/output'
out_prefix = 'excel_3months_enriched_2026Q1'

print('sources =')
for s in sources:
    print(s)
print('sa_type =', sa_type)
print('mem_limit =', mem_limit)
print('output_dir =', output_dir)
print('out_prefix =', out_prefix)

In [ ]:
def normalize_numeric_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r"[eE][+-]?\d+$", s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r"\.0+$", '', s)
    s = re.sub(r"\D", '', s)
    if not s:
        return None
    if len(s) == 9:
        s = '0' + s
    elif len(s) == 11:
        s = '0' + s
    return s


def normalize_contract(v):
    if pd.isna(v):
        return None
    s = str(v).strip().lower()
    s = re.sub(r'\s+', '', s)
    return s if s else None

## 1) Load and unify 3 Excel reports (base 1:1 rows)

In [ ]:
frames = []
all_cols = set()
load_stats = []

for src in sources:
    report_month = src['report_month']
    excel_path = src['excel_path']

    header_row = int(src.get('header', 0))
    df = pd.read_excel(excel_path, header=header_row)
    df.columns = [str(c).strip() for c in df.columns]

    df['report_month'] = report_month
    if excel_col_inn in df.columns:
        df['inn_norm'] = df[excel_col_inn].apply(normalize_numeric_str)
    else:
        df['inn_norm'] = None

    if excel_col_contract in df.columns:
        df['contract_number_norm'] = df[excel_col_contract].apply(normalize_contract)
    else:
        df['contract_number_norm'] = None

    load_stats.append({
        'report_month': report_month,
        'excel_path': excel_path,
        'rows': len(df),
        'unique_inn_norm': int(df['inn_norm'].dropna().nunique()),
    })

    all_cols.update(df.columns.tolist())
    frames.append(df)

all_cols = ['report_month', 'inn_norm', 'contract_number_norm'] + sorted(
    [c for c in all_cols if c not in {'report_month', 'inn_norm', 'contract_number_norm'}]
)

aligned_frames = []
for df in frames:
    aligned_frames.append(df.reindex(columns=all_cols))

agg_df = pd.concat(aligned_frames, ignore_index=True)

load_stats_df = pd.DataFrame(load_stats)
display(load_stats_df)
print('total_rows =', len(agg_df))
print('total_unique_inn_norm =', agg_df['inn_norm'].dropna().nunique())
display(agg_df.head(30))

## 2) Connect Impala + load Lake attrs (INN filter, OGRN, CDI_ID, SSP)

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

with imp:
    imp.execute(f"set MEM_LIMIT={mem_limit}")
    imp.execute('invalidate metadata ods_alpha.scd1_agreements')
    imp.execute('invalidate metadata ods_alpha.scd1_companies')
    imp.execute('invalidate metadata ods.scd1_z_r2_ip_merchants')
    imp.execute('invalidate metadata ods.scd1_z_cl_corp')
    imp.execute('invalidate metadata ocrm_ul.s_org_ext')
    imp.execute('invalidate metadata cdiul.ext_id_org')

print('Impala initialized')


def to_sql_in_list(values):
    out = []
    for v in values:
        s = str(v).replace("'", "''")
        out.append("'" + s + "'")
    return ', '.join(out)


def chunk_list(values, chunk_size):
    for i in range(0, len(values), chunk_size):
        yield values[i:i + chunk_size]


def fetch_in_chunks(values, chunk_size, sql_builder):
    out_df = pd.DataFrame()
    for part in chunk_list(values, chunk_size):
        in_list = to_sql_in_list(part)
        if not in_list:
            continue
        with imp:
            imp.execute(f"set MEM_LIMIT={mem_limit}")
            d = imp.fetch(sql_builder(in_list))
        if len(d):
            out_df = d if len(out_df) == 0 else pd.concat([out_df, d], ignore_index=True)
    return out_df

all_inn = sorted(agg_df['inn_norm'].dropna().unique().tolist())


def build_sql_lake_inn(in_list):
    return f"""
    select distinct regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn_norm
    from ods_alpha.scd1_agreements a
    join ods_alpha.scd1_companies c on c.n_cmp = a.n_cmp_client
    where regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') in ({in_list})
      and upper(trim(cast(a.acq_class as string))) = '{sa_type}'
      and cast(a.d_valid_from as date) <= cast('2026-03-01' as date)
      and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('2026-01-01' as date))
      and coalesce(a.ods_deleted_flg, '0') <> '1'
      and coalesce(c.ods_deleted_flg, '0') <> '1'
    """

lake_inn_df = fetch_in_chunks(all_inn, 700, build_sql_lake_inn)
lake_inn_df['inn_norm'] = lake_inn_df['inn_norm'].apply(normalize_numeric_str)
lake_inn_set = set(lake_inn_df['inn_norm'].dropna().tolist())

agg_df_lake = agg_df[agg_df['inn_norm'].isin(lake_inn_set)].copy()


def build_sql_ogrn(in_list):
    return f"""
    with b as (
      select distinct
        regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn_norm,
        cast(a.abs_agr_id as string) as agr_id
      from ods_alpha.scd1_agreements a
      join ods_alpha.scd1_companies c on c.n_cmp = a.n_cmp_client
      where regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') in ({in_list})
        and upper(trim(cast(a.acq_class as string))) = '{sa_type}'
        and cast(a.d_valid_from as date) <= cast('2026-03-01' as date)
        and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('2026-01-01' as date))
        and a.abs_agr_id is not null
    )
    select b.inn_norm, cast(corp.c_register_gos_reg_num_rec as string) as ogrn
    from b
    left join ods.scd1_z_r2_ip_merchants m on cast(m.id as string) = b.agr_id
    left join ods.scd1_z_cl_corp corp on corp.id = m.c_cl_org
    """

ogrn_df = fetch_in_chunks(sorted(list(lake_inn_set)), 700, build_sql_ogrn) if len(lake_inn_set) else pd.DataFrame(columns=['inn_norm','ogrn'])
if len(ogrn_df):
    ogrn_df['inn_norm'] = ogrn_df['inn_norm'].apply(normalize_numeric_str)
    ogrn_df['ogrn'] = ogrn_df['ogrn'].astype(str).str.strip()
    ogrn_df.loc[ogrn_df['ogrn'].isin(['', 'None', 'nan']), 'ogrn'] = None
    ogrn_df = ogrn_df.drop_duplicates('inn_norm')[['inn_norm', 'ogrn']].reset_index(drop=True)


def build_sql_ocrm(in_list):
    return f"""
    with t as (
      select
        cast(row_id as string) as row_id,
        regexp_replace(trim(cast(x_inn as string)), '[^0-9]', '') as inn_norm,
        case
          when lower(trim(cast(x_area_resp as string))) like 'дммб%'
            or lower(trim(cast(x_area_resp as string))) like 'дмсб (ми%'
            then 'ДМ'
          when lower(trim(cast(x_area_resp as string))) like 'дмсб (ма%'
            then 'ДМСБ'
          when lower(trim(cast(x_area_resp as string))) like 'дмсб (ср%'
            or lower(trim(cast(x_area_resp as string))) like 'дсб%'
            then 'ДМСБ'
          when lower(trim(cast(x_area_resp as string))) like 'дкб%'
            then 'ДКБ'
          when lower(trim(cast(x_area_resp as string))) like 'дрпа%'
            then 'ДРПА'
          when lower(trim(cast(x_area_resp as string))) like 'не под%'
            then 'Не подлежит сегментации'
          when nullif(trim(cast(x_area_resp as string)), '') is null then null
          else trim(cast(x_area_resp as string))
        end as ssp_ocrm,
        created,
        row_number() over (
          partition by regexp_replace(trim(cast(x_inn as string)), '[^0-9]', '')
          order by created desc, row_id desc
        ) as rn
      from ocrm_ul.s_org_ext
      where x_removed_flg = 'N'
        and x_duplicate_flg = 'N'
        and regexp_replace(trim(cast(x_inn as string)), '[^0-9]', '') in ({in_list})
    ),
    m as (
      select distinct cast(party_id as string) as cdi_id, cast(cmo_ext_party_source_id as string) as row_id
      from cdiul.ext_id_org
      where cmo_ext_source_system like 'OCRM%'
    )
    select t.inn_norm, m.cdi_id, t.ssp_ocrm
    from t
    left join m on m.row_id = t.row_id
    where t.rn = 1
    """

ocrm_df = fetch_in_chunks(sorted(list(lake_inn_set)), 700, build_sql_ocrm) if len(lake_inn_set) else pd.DataFrame(columns=['inn_norm','cdi_id','ssp_ocrm'])
if len(ocrm_df):
    ocrm_df['inn_norm'] = ocrm_df['inn_norm'].apply(normalize_numeric_str)
    ocrm_df = ocrm_df.drop_duplicates('inn_norm')[['inn_norm', 'cdi_id', 'ssp_ocrm']].reset_index(drop=True)

if len(ogrn_df):
    agg_df_lake = agg_df_lake.merge(ogrn_df, on='inn_norm', how='left')
if len(ocrm_df):
    agg_df_lake = agg_df_lake.merge(ocrm_df, on='inn_norm', how='left')

print('rows_after_lake_filter =', len(agg_df_lake))
print('unique_inn_after_lake_filter =', agg_df_lake['inn_norm'].dropna().nunique())
display(agg_df_lake.head(30))

## 3) QC summary (enriched dataset)

In [ ]:
qc_df = pd.DataFrame([
    {'metric': 'rows_before_lake_filter', 'value': int(len(agg_df))},
    {'metric': 'rows_after_lake_filter', 'value': int(len(agg_df_lake))},
    {'metric': 'unique_inn_before', 'value': int(agg_df['inn_norm'].dropna().nunique())},
    {'metric': 'unique_inn_after', 'value': int(agg_df_lake['inn_norm'].dropna().nunique())},
    {'metric': 'ssp_fill_rate', 'value': float(agg_df_lake['ssp_ocrm'].notna().mean()) if 'ssp_ocrm' in agg_df_lake.columns and len(agg_df_lake) else None},
    {'metric': 'ogrn_fill_rate', 'value': float(agg_df_lake['ogrn'].notna().mean()) if 'ogrn' in agg_df_lake.columns and len(agg_df_lake) else None},
    {'metric': 'cdi_id_fill_rate', 'value': float(agg_df_lake['cdi_id'].notna().mean()) if 'cdi_id' in agg_df_lake.columns and len(agg_df_lake) else None},
])

display(qc_df)

display(
    agg_df_lake.groupby('report_month', as_index=False)
    .agg(rows=('report_month', 'size'), unique_inn_norm=('inn_norm', 'nunique'))
    .sort_values('report_month')
)

## 4) Export enriched aggregated dataset

In [ ]:
out_dir = Path(output_dir)
out_dir.mkdir(parents=True, exist_ok=True)

out_xlsx = out_dir / f"{out_prefix}.xlsx"
out_csv = out_dir / f"{out_prefix}.csv"
out_parquet = out_dir / f"{out_prefix}.parquet"

agg_df_lake.to_excel(out_xlsx, index=False)
agg_df_lake.to_csv(out_csv, index=False)
agg_df_lake.to_parquet(out_parquet, index=False)

print('Saved files:')
print(out_xlsx)
print(out_csv)
print(out_parquet)